In [ ]:
import pygsheets
from selenium import webdriver
from time import sleep
import pandas as pd


In [ ]:
client = pygsheets.authorize(service_account_file=r'C:\Users\fumio\Documents\gcp-project-365616-6b0f83cca4c2.json')

ss = client.open_by_url('https://docs.google.com/spreadsheets/d/1H-otjPW382f7cUNvaC_p5BJCMw7nmlmuhu2JiNBx_Y0/edit?gid=195125089#gid=195125089')

ws = ss.worksheet_by_title('Automação GeoSampa')

In [ ]:
df = ws.get_as_df()

In [ ]:
df = df[df['Proprietário'] == '']

In [ ]:
df.head()

In [ ]:
driver = webdriver.Chrome()
driver.get("https://notcertiptu.prefeitura.sp.gov.br/PaginasRestritas/frm001_Gerar_Notif_Lanc.aspx")

In [ ]:
def get_proprietario(driver: webdriver.Chrome, iptu: str) -> str:
    input_iptu = driver.find_element(by='id', value = 'txt_SQL')
    input_iptu.clear()
    input_iptu.send_keys(iptu.replace('.', '').replace('-', ''))

    input_ano = driver.find_element(by='id', value = 'txt_Exercicio')
    input_ano.clear()
    input_ano.send_keys('2024')

    button_consultar = driver.find_element(by='id', value = 'btnConsultar')
    button_consultar.click()

    errors_message = driver.find_elements(by = 'id', value = 'popup_ok')

    if len(errors_message) > 0:
        errors_message[0].click()
        return 'NAO ENCONTRADO'

    proprietario = driver.find_element(by='id', value='lblproprietario')
    proprietario = proprietario.text
    
    button_gerar = driver.find_element(by='id', value = 'btnGerar')
    button_gerar.click()

    sleep(1)
    try:
        cpf = driver.find_element(by='id', value = 'TxtProprietario')
        cpf = cpf.get_attribute('value')
    except:
        cpf = 'Não Identificado'
    finally:
        button_cancelar = driver.find_element(by='id', value = 'btnCancelar')
        button_cancelar.click()
        sleep(1)

    return f"{proprietario} - {cpf}"

In [ ]:
dicionario_iptu_proprietario = {}

In [ ]:
filtered_iptu = [iptu for iptu in df['IPTU'].tolist() if iptu not in dicionario_iptu_proprietario]

In [ ]:
for iptu in filtered_iptu:
    if filtered_iptu.index(iptu) % 10 == 0:
        print(round(filtered_iptu.index(iptu)/len(filtered_iptu) * 100, 1))
    dicionario_iptu_proprietario[iptu] = get_proprietario(driver, iptu)

In [ ]:
df['Proprietário'] = df['IPTU'].apply(lambda x: dicionario_iptu_proprietario.get(x, ''))

In [ ]:
filtros = [
    'banc',
    'caixa',
    'hsbc',
    'itau',
    'const',
    'empre',
    'imob',
    'admin',
    'cons',
    'igreja',
    'ltd',
    'nao encontrado',
    'hipoteca',
]

In [ ]:
df['Pessoa'] = df['Proprietário'].apply(lambda x: not any([filtro.upper() in str(x).upper() for filtro in filtros]))

In [ ]:
df.head()

In [ ]:
ws.set_dataframe(
    df,
    'A1'
)

In [ ]:
driver.close()

## Agrupando Proprietários

In [ ]:
ws = ss.worksheet_by_title('Automação GeoSampa')
df = ws.get_as_df()

In [ ]:
len(df)

In [ ]:
df = df[df['Pessoa'] == 'TRUE']

In [ ]:
len(df)

In [ ]:
filtros = [
    'Condomínio Ext Praça Morumbi',
    # 'Cristais da Terra - Panamby',
    # 'Reserva Morumbi',
    # 'Terras da Mata',
    # 'Mais Morumbi Clube',
    # 'Passarim',
    # 'Antigua Morumbi',
    # 'Condomínio Luiza',
]

In [ ]:
df = df[df['Condomínio'].isin(filtros)]

In [ ]:
len(df)

In [ ]:
grouped = df

grouped = grouped[grouped['Complemento'].str.contains("AP")]

grouped['Info'] = grouped.apply(lambda x: f"{x['Condomínio']} - {x['Complemento']}", axis = 1)

grouped = grouped.groupby('Proprietário')

concatenated_genders = grouped['Info'].apply(lambda x: ', '.join(sorted(set(x)))).reset_index()

sum_of_ages = grouped['Condomínio'].count().reset_index()

result = pd.merge(concatenated_genders, sum_of_ages, on='Proprietário')

In [ ]:
result

In [ ]:
dicionario_nome_cpf = {info.split(' - ')[0] : info for info in set(df['Proprietário'].tolist())}

In [ ]:
ws = ss.worksheet_by_title('Nomes Para SeekLoc')

In [ ]:
# df = ws.get_as_df()

In [ ]:
# df

In [ ]:
# df['Proprietário2'] = df['Proprietário'].apply(lambda x: dicionario_nome_cpf.get(x, ''))

In [ ]:
# df = df[['Proprietário2']]

In [ ]:
ws.insert_rows(
    1,
    len(result),
    result.values.tolist(),
)

In [ ]:
# ws.set_dataframe(
#     df,
#     'A1'
# )

{
    "filter": {
        "_and": [
            {
                "status_documento": {
                    "_eq": "Vigente"
                }
            },
            {
                "_and": [
                    {
                        "data_fim": {
                            "_lte": "$NOW(+15 days)"
                        }
                    },
                    {
                        "data_fim": {
                            "_gt": "$NOW(+14 days)"
                        }
                    }
                ]
            }
        ]
    }
}

In [ ]:
{
    "filter": {
        "status_documento": {
            "_eq": "Vigente"
        }
    }
}